# K-Means Clustering — Demo

End-to-end walkthrough on self-contained synthetic data + a real dataset.

**Pipeline:**
1. Synthetic 2D data to see how K-Means works visually
2. Scaling matters — see what happens without it
3. Elbow method to choose K
4. Silhouette score to validate
5. Effect of initialisation
6. Outlier sensitivity
7. PCA + K-Means on real high-dimensional data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import make_blobs, load_iris
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

np.random.seed(42)

## 1 — Synthetic Data with Clear Clusters

`make_blobs` creates well-separated 2D clusters — perfect for seeing what K-Means does.

In [ ]:
X, y_true = make_blobs(n_samples=300, centers=4, cluster_std=0.7, random_state=42)

plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], s=30, alpha=0.7)
plt.title('Raw data — no labels')
plt.xlabel('Feature 1'); plt.ylabel('Feature 2')
plt.show()

## 2 — Fit K-Means with K=4

We know there are 4 clusters (we made the data). Let's see if K-Means finds them.

In [ ]:
kmeans = KMeans(n_clusters=4, random_state=42, n_init=10).fit(X)

labels = kmeans.labels_
centroids = kmeans.cluster_centers_

plt.figure(figsize=(8, 6))
plt.scatter(X[:, 0], X[:, 1], c=labels, cmap='viridis', s=30, alpha=0.7)
plt.scatter(centroids[:, 0], centroids[:, 1], c='red', marker='X', s=200, label='Centroids')
plt.title(f'K-Means clusters (K=4)  —  WCSS = {kmeans.inertia_:.1f}')
plt.legend()
plt.show()

print(f'WCSS / inertia: {kmeans.inertia_:.2f}')
print(f'Iterations:     {kmeans.n_iter_}')
print(f'Silhouette:     {silhouette_score(X, labels):.3f}')

## 3 — Why Scaling Matters

Let's manually scale one feature 100× and watch what happens — the cluster structure changes because that feature dominates the distance.

In [ ]:
# Multiply feature 1 by 100 to simulate unequal scales
X_unbalanced = X.copy()
X_unbalanced[:, 0] = X_unbalanced[:, 0] * 100

km_unb = KMeans(n_clusters=4, random_state=42, n_init=10).fit(X_unbalanced)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(X_unbalanced[:, 0], X_unbalanced[:, 1], c=km_unb.labels_, cmap='viridis', s=30, alpha=0.7)
axes[0].set_title('Without scaling — clusters dominated by feature 1')

# Now scale properly
scaler = StandardScaler()
X_unb_scaled = scaler.fit_transform(X_unbalanced)
km_scaled = KMeans(n_clusters=4, random_state=42, n_init=10).fit(X_unb_scaled)
axes[1].scatter(X_unb_scaled[:, 0], X_unb_scaled[:, 1], c=km_scaled.labels_, cmap='viridis', s=30, alpha=0.7)
axes[1].set_title('After StandardScaler — proper clusters')
plt.tight_layout()
plt.show()

## 4 — Elbow Method to Choose K

Pretend we don't know K. Plot WCSS for K = 1 to 10.

In [ ]:
inertias = []
K_range = range(1, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)
    inertias.append(km.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(K_range, inertias, 'o-')
plt.xlabel('K (number of clusters)')
plt.ylabel('WCSS / inertia')
plt.title('Elbow method')
plt.axvline(4, color='red', linestyle='--', alpha=0.5, label='True K = 4')
plt.legend()
plt.grid(alpha=0.3)
plt.show()

The curve bends sharply at K=4 — that's the elbow. Beyond 4, WCSS keeps dropping slowly but adds no real structure.

## 5 — Silhouette Score for K Selection

Compute silhouette score for K = 2 to 10 (silhouette undefined for K=1).

In [ ]:
sil_scores = []
K_range = range(2, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X)
    score = silhouette_score(X, km.labels_)
    sil_scores.append(score)
    print(f'K={k}: silhouette = {score:.3f}')

best_k = K_range[np.argmax(sil_scores)]
print(f'\nBest K by silhouette: {best_k}')

plt.figure(figsize=(8, 5))
plt.plot(K_range, sil_scores, 'o-', color='green')
plt.xlabel('K')
plt.ylabel('Silhouette score')
plt.title('Silhouette analysis')
plt.grid(alpha=0.3)
plt.show()

Both elbow and silhouette agree on K=4. That's the win.

## 6 — Effect of Random Initialisation

Run K-Means with `init='random'` and a fixed-but-bad seed to see how unstable it can be. With `n_init=1`, we don't get the safety of multiple tries.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for i, seed in enumerate([0, 1, 5]):
    km = KMeans(n_clusters=4, init='random', n_init=1, random_state=seed).fit(X)
    axes[i].scatter(X[:, 0], X[:, 1], c=km.labels_, cmap='viridis', s=30, alpha=0.7)
    axes[i].scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1], c='red', marker='X', s=150)
    axes[i].set_title(f'Random init, seed={seed}, WCSS={km.inertia_:.1f}')
plt.tight_layout()
plt.show()

Different seeds can give different results — sometimes the algorithm gets stuck in local minima. **K-means++ with `n_init=10` is the safe default.**

## 7 — Outliers Distort Clusters

In [ ]:
# Add a few extreme outliers
X_with_outliers = np.vstack([X, [[20, 20], [-20, -20], [25, -25]]])

km_out = KMeans(n_clusters=4, random_state=42, n_init=10).fit(X_with_outliers)

plt.figure(figsize=(9, 6))
plt.scatter(X_with_outliers[:, 0], X_with_outliers[:, 1], c=km_out.labels_, cmap='viridis', s=30, alpha=0.7)
plt.scatter(km_out.cluster_centers_[:, 0], km_out.cluster_centers_[:, 1], c='red', marker='X', s=200, label='Centroids')
plt.title('K-Means with outliers — centroids dragged toward extremes')
plt.legend()
plt.show()

Notice how centroids have been **pulled away** from the dense clusters by the outliers. The clusters look distorted.

Remove or transform outliers before clustering for robust results.

## 8 — Real Data: Iris Dataset with PCA

Iris is a 4-feature dataset. Use PCA to reduce to 2 features for visualisation.

In [ ]:
data = load_iris()
X_iris = data.data
y_iris = data.target

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_iris)

# Reduce 4D → 2D with PCA
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)
print(f'Variance retained: {pca.explained_variance_ratio_.sum() * 100:.1f}%')

In [ ]:
# Cluster in the reduced space
km_iris = KMeans(n_clusters=3, random_state=42, n_init=10).fit(X_scaled)
labels_pred = km_iris.labels_

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].scatter(X_pca[:, 0], X_pca[:, 1], c=y_iris, cmap='viridis', s=40, alpha=0.7)
axes[0].set_title('True species labels')
axes[1].scatter(X_pca[:, 0], X_pca[:, 1], c=labels_pred, cmap='viridis', s=40, alpha=0.7)
axes[1].set_title('K-Means predicted clusters')
for ax in axes:
    ax.set_xlabel('PC1'); ax.set_ylabel('PC2')
plt.tight_layout()
plt.show()

print(f'Silhouette: {silhouette_score(X_scaled, labels_pred):.3f}')

K-Means recovers Iris species reasonably well — even without using the labels. Two species (versicolor and virginica) overlap, but setosa is clean.

## 9 — Iris Elbow + Silhouette

In [ ]:
iris_results = []
for k in range(1, 11):
    km = KMeans(n_clusters=k, random_state=42, n_init=10).fit(X_scaled)
    sil = silhouette_score(X_scaled, km.labels_) if k > 1 else None
    iris_results.append({'K': k, 'WCSS': round(km.inertia_, 2), 'silhouette': round(sil, 3) if sil else None})

print(pd.DataFrame(iris_results))

## Key Learnings

1. **K-Means partitions data into K clusters** — assign points to nearest centroid, update centroids, repeat.

2. **Always scale features first** — distance-based methods are scale-sensitive.

3. **Choose K with elbow + silhouette** — combine both metrics; sanity-check with domain knowledge.

4. **Use K-means++ with `n_init=10`** — sklearn's defaults handle initialisation correctly.

5. **Outliers distort centroids** — remove or transform them first.

6. **For high-dimensional data, reduce first** — PCA + K-Means is a common pipeline.

7. **Silhouette score validates cluster quality** — > 0.5 is reasonable, > 0.7 is strong.

8. **K-Means assumes spherical clusters** — for irregular shapes use DBSCAN; for categorical data use K-Modes.